# Assignment 1: Weather Agent

A Gemini SDK agent that uses the OpenWeather API to get the current weather for **Kathmandu, London and Tokyo**, making the tool calls **sequentially**, then calculates and displays the **average temperature (°C)** in Python.

This is the standalone copy of the solution from `GenAI01_gemini_ai.ipynb`. The "Level" references below point to sections of that course notebook. API keys are loaded from the project's `.env` file.

## ✅ Assignment 1 Solution — Weather Agent

**Goal:** a Gemini agent looks up the current weather for **Kathmandu, London and Tokyo** (OpenWeather), **one location at a time**, and the program shows the **average temperature (°C)**.

**How it works**

```
prompt ──▶ Gemini ──"call get_current_weather('Kathmandu,NP')"──▶ Python runs the real OpenWeather request
               ▲                                                         │
               └──────────────── tool result sent back ◀─────────────────┘
         (repeats for London, then Tokyo) ──▶ Gemini writes a final summary
```

**Who does what**
- **Gemini** decides which tool call to make next and writes the final summary.
- **Python** makes the real HTTP requests, records every reading, and calculates the average — so the number is always based on actual OpenWeather data.

**Concepts reused from this notebook**
- **Level 5:** a normal Python function with type hints + docstring becomes a Gemini tool.
- **Level 6:** *manual* function calling — automatic function calling is disabled, we read `response.function_calls`, run the function ourselves, and send the result back with `Part.from_function_response`.
- **Level 7:** sequential tool calls — the loop repeats until Gemini has fetched all three locations, in order.

**Rule for the average:** it is only shown when **all three** locations return a valid temperature. If any location fails, the failure is reported and no (misleading) partial average is calculated.

Run the cells below in order: **B** (setup) → **C** (tool) → **D** (tool test) → **E** (agent) → **F** (helpers) → **G** (run & report).

In [ ]:
import os

import requests
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

load_dotenv()

MODEL_ID = "gemini-3.6-flash"
LOCATIONS = ["Kathmandu,NP", "London,GB", "Tokyo,JP"]  # "City,CountryCode"
OPENWEATHER_URL = "https://api.openweathermap.org/data/2.5/weather"
UNITS = "metric"               # "metric" -> temperatures in degrees Celsius
REQUEST_TIMEOUT_SECONDS = 10   # give up on a slow OpenWeather request
MAX_AGENT_TURNS = 6            # safety limit so the agent loop can't run forever

REQUIRED_KEYS = ["GEMINI_API_KEY", "OPENWEATHER_API_KEY"]
missing_keys = [name for name in REQUIRED_KEYS if not os.getenv(name)]

for name in REQUIRED_KEYS:
    print(f"{name}: {'missing ✗' if name in missing_keys else 'found ✓'}")

if missing_keys:
    raise RuntimeError(f"Missing API key(s) in .env: {', '.join(missing_keys)}")

# ---- The assignment requires exactly three locations ---------------------
if len(LOCATIONS) != 3:
    raise ValueError(f"The assignment needs exactly 3 locations, but {len(LOCATIONS)} are configured.")

# Create the Gemini client (the key is passed in, never printed).
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print(f"Model: {MODEL_ID}")
print(f"Locations (in order): {', '.join(LOCATIONS)}")

GEMINI_API_KEY: found ✓
OPENWEATHER_API_KEY: found ✓
Model: gemini-3.6-flash
Locations (in order): Kathmandu,NP, London,GB, Tokyo,JP


In [10]:
def get_current_weather(city: str) -> dict:
    """Gets the current weather for one location from the OpenWeather API.

    Call this tool once per location. Temperatures are in degrees Celsius.

    Args:
        city: The location as "City,CountryCode", for example "Kathmandu,NP".
    """
    city = (city or "").strip()

    def failure(message: str) -> dict:
        return {"ok": False, "requested_location": city, "error": message}

    if not city:
        return failure("No location was given.")

    api_key = os.getenv("OPENWEATHER_API_KEY")
    if not api_key:
        return failure("OPENWEATHER_API_KEY is not set.")

    params = {"q": city, "appid": api_key, "units": UNITS}
    try:
        response = requests.get(OPENWEATHER_URL, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
    except requests.exceptions.Timeout:
        return failure(f"OpenWeather did not respond within {REQUEST_TIMEOUT_SECONDS} seconds.")
    except requests.exceptions.ConnectionError:
        return failure("Could not connect to OpenWeather (check your internet connection).")
    except requests.exceptions.RequestException:
        return failure("The request to OpenWeather failed.")

    if response.status_code == 401:
        return failure("OpenWeather rejected the API key (HTTP 401).")
    if response.status_code == 404:
        return failure(f"Location not found by OpenWeather: '{city}' (HTTP 404).")
    if response.status_code == 429:
        return failure("OpenWeather rate limit reached (HTTP 429). Try again shortly.")
    if response.status_code != 200:
        return failure(f"OpenWeather returned an unexpected status (HTTP {response.status_code}).")

    try:
        data = response.json()
    except ValueError:
        return failure("OpenWeather returned a response that is not valid JSON.")

    try:
        temperature = data["main"]["temp"]
        city_name = data["name"]
        country = data.get("sys", {}).get("country", "")
        description = data["weather"][0]["description"]
    except (KeyError, IndexError, TypeError, AttributeError):
        return failure("OpenWeather's response was missing expected weather fields.")

    # bool is a subclass of int in Python, so rule it out explicitly.
    if isinstance(temperature, bool) or not isinstance(temperature, (int, float)):
        return failure("OpenWeather returned a temperature that is not a number.")

    return {
        "ok": True,
        "requested_location": city,
        "city": city_name,
        "country": country,
        "temperature_c": float(temperature),
        "description": description,
    }

In [5]:
from pprint import pprint

print("--- Test 1: a real location ---")
pprint(get_current_weather("Kathmandu,NP"))

print("\n--- Test 2: a location that doesn't exist (should return an error, not crash) ---")
pprint(get_current_weather("NotARealCity123"))

--- Test 1: a real location ---
{'city': 'Kathmandu',
 'country': 'NP',
 'description': 'light rain',
 'ok': True,
 'requested_location': 'Kathmandu,NP',
 'temperature_c': 20.01}

--- Test 2: a location that doesn't exist (should return an error, not crash) ---
{'error': "Location not found by OpenWeather: 'NotARealCity123' (HTTP 404).",
 'ok': False,
 'requested_location': 'NotARealCity123'}


In [ ]:
SYSTEM_INSTRUCTION = (
    "You are a weather data agent with one tool: get_current_weather.\n"
    "Rules:\n"
    "1. Call get_current_weather exactly once for each location the user lists.\n"
    "2. Work sequentially: request only ONE location per turn, in the exact order given, "
    "and wait for its result before requesting the next.\n"
    "3. Pass each location string exactly as the user wrote it (for example 'Kathmandu,NP').\n"
    "4. Never invent or estimate temperatures. If a tool result has ok=false, "
    "report that location's error plainly.\n"
    "5. Do not calculate an average; the program calculates it from the tool results.\n"
    "6. When every location is done, give a short summary of each location's weather."
)

AGENT_CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM_INSTRUCTION,
    tools=[get_current_weather],   # the SDK builds the tool schema from the function
    temperature=0.0,               # predictable, rule-following behaviour
    # Crucial (Level 6): disable automatic calling so Gemini hands the call back to us.
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)

# Map tool names (as Gemini sends them) to the real Python functions (Level 6).
available_tools = {"get_current_weather": get_current_weather}


def run_weather_agent(locations: list[str]) -> tuple[list[dict], str]:
    """Runs the manual function-calling loop.

    Returns:
        (readings, final_text): every tool result in the order it happened,
        and Gemini's final text summary (or a clear "[Agent stopped] ..." message).
    """
    readings: list[dict] = []
    tool_call_number = 0

    ordered_list = "; ".join(f"{i}. {loc}" for i, loc in enumerate(locations, start=1))
    user_prompt = f"Get the current weather for these locations, one at a time, in this order: {ordered_list}"
    print(f"User prompt: {user_prompt}\n")

    # The conversation history we manage ourselves (Level 6).
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=user_prompt)])]

    for turn in range(1, MAX_AGENT_TURNS + 1):
        # ---- Ask Gemini what to do next ---------------------------------
        try:
            response = client.models.generate_content(model=MODEL_ID, contents=contents, config=AGENT_CONFIG)
        except errors.APIError as exc:
            return readings, f"[Agent stopped] Gemini API error on turn {turn}: {exc.code} {exc.message}"
        except Exception as exc:  # e.g. network problems reaching Gemini
            return readings, f"[Agent stopped] Could not reach Gemini on turn {turn} ({type(exc).__name__})."

        if not response.candidates or response.candidates[0].content is None:
            return readings, f"[Agent stopped] Gemini returned no content on turn {turn}."

        # Keep Gemini's turn in the history exactly as returned (Level 6).
        contents.append(response.candidates[0].content)

        # ---- No tool requested -> Gemini has given its final answer -------
        function_calls = response.function_calls
        if not function_calls:
            return readings, (response.text or "(Gemini returned no text summary.)")

        if len(function_calls) > 1:
            print(f"(Turn {turn}: Gemini requested {len(function_calls)} calls at once — running them one at a time.)")

        # ---- Run each requested call, strictly one after another ----------
        response_parts = []
        for call in function_calls:
            tool_call_number += 1
            args = dict(call.args or {})
            print(f"[TOOL CALL {tool_call_number}] turn {turn}: {call.name}({args})")

            tool_function = available_tools.get(call.name)
            if tool_function is None:
                result = {"ok": False, "error": f"Unknown tool '{call.name}'."}
            else:
                try:
                    result = tool_function(**args)
                except TypeError:
                    result = {"ok": False, "requested_location": "", "error": f"Invalid arguments for {call.name}: {args}"}
                readings.append(result)

            if result.get("ok"):
                print(f"    → {result['city']}, {result['country']}: "
                      f"{result['temperature_c']:.2f} °C ({result['description']})")
            else:
                print(f"    → ERROR: {result['error']}")

            response_parts.append(types.Part.from_function_response(name=call.name, response=result))

        # Send all results for this turn back together, as one message.
        contents.append(types.Content(role="user", parts=response_parts))

    return readings, f"[Agent stopped] Reached the safety limit of {MAX_AGENT_TURNS} turns before Gemini finished."

In [ ]:
def _city_key(location: str) -> str:
    """'Kathmandu,NP' -> 'kathmandu'. Used to match locations to readings,
    even if Gemini passes 'Kathmandu' instead of 'Kathmandu,NP'."""
    return (location or "").split(",")[0].strip().lower()


def summarize_readings(locations: list[str], readings: list[dict]) -> dict:
    """Matches each expected location (in order) to what the agent actually fetched.

    Returns:
        {"results": one entry per expected location,
         "valid_temperatures": temperatures from successful readings,
         "all_valid": True only if every location has a valid reading}
    """
    results = []
    for location in locations:
        matches = [r for r in readings if _city_key(r.get("requested_location", "")) == _city_key(location)]
        successes = [r for r in matches if r.get("ok")]

        if successes:
            results.append({"location": location, **successes[-1]})
        elif matches:
            results.append({"location": location, **matches[-1]})
        else:
            results.append({"location": location, "ok": False,
                            "error": "The agent never requested this location."})

    valid_temperatures = [r["temperature_c"] for r in results if r.get("ok")]
    return {
        "results": results,
        "valid_temperatures": valid_temperatures,
        "all_valid": len(valid_temperatures) == len(locations),
    }


def calculate_average_temperature(temperatures: list[float]) -> float:
    """Plain arithmetic mean: sum of the values divided by how many there are."""
    if not temperatures:
        raise ValueError("Cannot average an empty list of temperatures.")
    return sum(temperatures) / len(temperatures)

In [8]:
print(f"=== Running weather agent ({MODEL_ID}) ===\n")
readings, gemini_summary = run_weather_agent(LOCATIONS)
summary = summarize_readings(LOCATIONS, readings)

print("\n=== Weather Report (°C) ===")
for number, r in enumerate(summary["results"], start=1):
    if r.get("ok"):
        place = f"{r['city']}, {r['country']}"
        print(f"{number}. {place:<16} {r['temperature_c']:>7.2f} °C   {r['description']}")
    else:
        print(f"{number}. {r['location']:<16} FAILED: {r['error']}")

valid_count = len(summary["valid_temperatures"])
total = len(LOCATIONS)

if summary["all_valid"]:
    average = calculate_average_temperature(summary["valid_temperatures"])
    working = " + ".join(f"{t:.2f}" for t in summary["valid_temperatures"])
    print(f"\nCalculation: ({working}) / {valid_count}")
    print(f"Average temperature ({valid_count} of {total} locations): {average:.2f} °C")
else:
    print(f"\nAverage NOT calculated: only {valid_count} of {total} locations returned valid temperatures.")
    print("A partial average would be misleading, so none is shown.")

print("\n--- Gemini's summary (the Python numbers above are the authoritative ones) ---")
print(gemini_summary)

=== Running weather agent (gemini-3.6-flash) ===

User prompt: Get the current weather for these locations, one at a time, in this order: 1. Kathmandu,NP; 2. London,GB; 3. Tokyo,JP

[TOOL CALL 1] turn 1: get_current_weather({'city': 'Kathmandu,NP'})
    → Kathmandu, NP: 20.01 °C (light rain)
[TOOL CALL 2] turn 2: get_current_weather({'city': 'London,GB'})
    → London, GB: 11.90 °C (overcast clouds)
[TOOL CALL 3] turn 3: get_current_weather({'city': 'Tokyo,JP'})
    → Tokyo, JP: 24.02 °C (broken clouds)

=== Weather Report (°C) ===
1. Kathmandu, NP      20.01 °C   light rain
2. London, GB         11.90 °C   overcast clouds
3. Tokyo, JP          24.02 °C   broken clouds

Calculation: (20.01 + 11.90 + 24.02) / 3
Average temperature (3 of 3 locations): 18.64 °C

--- Gemini's summary (the Python numbers above are the authoritative ones) ---
Here is the current weather summary for each requested location:

- **Kathmandu, NP**: 20.01°C, light rain
- **London, GB**: 11.9°C, overcast clouds
- 